In [3]:
#Carga el dataset 
from datasets import load_dataset

dataset = load_dataset("beltrewilton/punta-cana-spanish-reviews")


In [4]:
#Exploración de los datos 
print(dataset)
print(dataset["train"].column_names)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['hotel_name', 'location', 'wrote', 'rating', 'title', 'review_text'],
        num_rows: 34561
    })
})
['hotel_name', 'location', 'wrote', 'rating', 'title', 'review_text']
{'hotel_name': 'Secrets Cap Cana Resort & Spa', 'location': 'Villaviciosa, Spain', 'wrote': 'December 2021', 'rating': 5, 'title': 'Vacaciones en El paraíso', 'review_text': 'El hotel impecable todo muy limpio las habitaciones amplias y con todo tipo de detalles. Las medidas cocid estaban bien ya que en el bufete tú no tocas nada todo te lo sirven ellos . En la piscina y en la playa hay muchísimos camareros muy atentos , en especial Fidelina( es la bomba) que te sirven continuamente y te desinfectan las manos antes de coger la bebida.Los restaurantes , la barbacoa de la playa con la paella y el bufete de diez, no puedes decir que algo está malo.Todo el personal es súper amable y divertido.Nuestro mayordomo Gerlyn, un diez . Lo mejor LA DIVA, que junto a sus comp

In [5]:
#Transformación a df 
train_df = dataset["train"].to_pandas()

train_df.head()

,hotel_name,location,wrote,rating,title,review_text
0,Secrets Cap Cana Resort & Spa,"Villaviciosa, Spain",December 2021,5,Vacaciones en El paraíso,El hotel impecable todo muy limpio las habitac...
1,Secrets Cap Cana Resort & Spa,"Villaviciosa, Spain",December 2021,5,HOTEL de la EXCELENCIA,"EXCELENTE, magnífico, excepcional, maravilloso..."
2,Secrets Cap Cana Resort & Spa,"Villaviciosa, Spain",December 2021,5,Sentirse como en CASA!!!!!,Tras conocer bastantes hoteles en distintos lu...
3,Secrets Cap Cana Resort & Spa,"Villaviciosa, Spain",December 2021,5,Muy buen nivel,Muy buen hotel. Cumple con todos los requisito...
4,Secrets Cap Cana Resort & Spa,"Villaviciosa, Spain",December 2021,5,10,Excelente todo ....sobre todo el personal. Es ...


In [6]:
#Convertimos la columna rating en una etiqueta numerica de sentimiento
def rating_to_label(rating):
    rating = int(rating)

    if rating <= 2:
        return 0   # NEG
    elif rating == 3:
        return 1   # NEU
    else:
        return 2   # POS

train_df["label"] = train_df["rating"].apply(rating_to_label)

In [7]:
train_df

,hotel_name,location,wrote,rating,title,review_text,label
0,Secrets Cap Cana Resort & Spa,"Villaviciosa, Spain",December 2021,5,Vacaciones en El paraíso,El hotel impecable todo muy limpio las habitac...,2
1,Secrets Cap Cana Resort & Spa,"Villaviciosa, Spain",December 2021,5,HOTEL de la EXCELENCIA,"EXCELENTE, magnífico, excepcional, maravilloso...",2
2,Secrets Cap Cana Resort & Spa,"Villaviciosa, Spain",December 2021,5,Sentirse como en CASA!!!!!,Tras conocer bastantes hoteles en distintos lu...,2
3,Secrets Cap Cana Resort & Spa,"Villaviciosa, Spain",December 2021,5,Muy buen nivel,Muy buen hotel. Cumple con todos los requisito...,2
4,Secrets Cap Cana Resort & Spa,"Villaviciosa, Spain",December 2021,5,10,Excelente todo ....sobre todo el personal. Es ...,2
...,...,...,...,...,...,...,...
34556,Ocean El Faro,None,October 2019,5,Primera vez en este maravilloso Hotel,Definitivamente fue una excelente opción el se...,2
34557,Ocean El Faro,None,October 2019,5,Good,Hemos pasado unos dias fabulosos hay un trato ...,2
34558,Ocean El Faro,None,October 2019,5,Good,El hotel está muy bueno súper acogedor un pers...,2
34559,Ocean El Faro,None,October 2019,5,Cumpleaños celebración,"La calidad del hotel es primium, estuve en feb...",2


In [8]:
#Cantidad de registros por Clase
train_df.info()
train_df["label"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34561 entries, 0 to 34560
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   hotel_name   34561 non-null  object
 1   location     30190 non-null  object
 2   wrote        34514 non-null  object
 3   rating       34561 non-null  int64 
 4   title        34561 non-null  object
 5   review_text  34561 non-null  object
 6   label        34561 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 1.8+ MB


label
2    30632
0     2234
1     1695
Name: count, dtype: int64

In [9]:
#Concer la longitud de las reseñas
train_df["text_length"] = train_df["review_text"].astype(str).str.len()
train_df["text_length"].describe()

count    34561.000000
mean       435.609155
std        214.576992
min         60.000000
25%        246.000000
50%        354.000000
75%        630.000000
max        790.000000
Name: text_length, dtype: float64

In [11]:
#Dejando listo mis datos para entrenar
train_df = train_df.rename(columns={
    "review_text": "text"
})

train_df = train_df[["text", "label"]]

print(train_df.head())

                                                text  label
0  El hotel impecable todo muy limpio las habitac...      2
1  EXCELENTE, magnífico, excepcional, maravilloso...      2
2  Tras conocer bastantes hoteles en distintos lu...      2
3  Muy buen hotel. Cumple con todos los requisito...      2
4  Excelente todo ....sobre todo el personal. Es ...      2


MODELO ENTRENADO


In [14]:
from transformers import pipeline

model_name = "pysentimiento/robertuito-sentiment-analysis"

#usar el modelo
classifier = pipeline(
    "sentiment-analysis",
    model=model_name,
    tokenizer=model_name
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1902.86it/s]


In [15]:
texts = [
    "Me encantó el producto, llegó rápido y funciona perfecto.",
    "La atención fue pésima y nunca me respondieron.",
    "El paquete llegó ayer."
]

for text in texts:
    print(text)
    print(classifier(text))

Me encantó el producto, llegó rápido y funciona perfecto.
[{'label': 'POS', 'score': 0.9799479246139526}]
La atención fue pésima y nunca me respondieron.
[{'label': 'NEG', 'score': 0.9411347508430481}]
El paquete llegó ayer.
[{'label': 'NEU', 'score': 0.7872689366340637}]


FINETUNING


In [16]:

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

#Cargar y descarga modelo y tokenizador
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3664.41it/s]


In [17]:
train_df

,text,label
0,El hotel impecable todo muy limpio las habitac...,2
1,"EXCELENTE, magnífico, excepcional, maravilloso...",2
2,Tras conocer bastantes hoteles en distintos lu...,2
3,Muy buen hotel. Cumple con todos los requisito...,2
4,Excelente todo ....sobre todo el personal. Es ...,2
...,...,...
34556,Definitivamente fue una excelente opción el se...,2
34557,Hemos pasado unos dias fabulosos hay un trato ...,2
34558,El hotel está muy bueno súper acogedor un pers...,2
34559,"La calidad del hotel es primium, estuve en feb...",2


In [18]:
from datasets import Dataset

#Convertir df a dataset de Hugging Face
hf_dataset = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)

#Divido mis datos en entrenamiento y prueba
dataset = hf_dataset.train_test_split(test_size=0.2, seed=42)


#Funcion para tokenizar mis datos
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
#Aplicar tokenizador a mis datos
tokenized_dataset = dataset.map(tokenize, batched=True)

Map: 100%|██████████| 6913/6913 [00:01<00:00, 6120.38 examples/s]


In [19]:
#Me quedo solo con las columnas necesarias
columns_to_remove = [
    col for col in tokenized_dataset["train"].column_names
    if col not in ["input_ids", "attention_mask", "label"]
]

tokenized_dataset = tokenized_dataset.remove_columns(columns_to_remove)

#Convertir los datos que necesita PyTorch 
tokenized_dataset.set_format("torch")

tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 27648
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 6913
    })
})

In [20]:

#Cargar el modelo para que tenga la cantidad correcta de clases
num_labels = len(set(dataset["train"]["label"]))

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    ignore_mismatched_sizes=True
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3765.84it/s]


In [21]:
#CONFIGURACION DEL ENTRENAMIENTO
training_args = TrainingArguments(
    output_dir="../models/robertuito_finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

In [22]:
#FUNCION DE METRICAS
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [23]:
from datasets import concatenate_datasets

train_data = tokenized_dataset["train"]

samples_per_class = 500

balanced_parts = []

for label in [0, 1, 2]:
    class_data = train_data.filter(lambda x: x["label"] == label)
    class_data = class_data.shuffle(seed=42).select(range(samples_per_class))
    balanced_parts.append(class_data)

small_train = concatenate_datasets(balanced_parts).shuffle(seed=42)

Filter:   0%|          | 0/27648 [00:00<?, ? examples/s]

Filter: 100%|██████████| 27648/27648 [00:00<00:00, 30811.35 examples/s]


In [24]:
eval_data = tokenized_dataset["test"]

eval_parts = []

for label in [0, 1, 2]:
    class_data = eval_data.filter(lambda x: x["label"] == label)
    class_data = class_data.shuffle(seed=42).select(range(150))
    eval_parts.append(class_data)

small_eval = concatenate_datasets(eval_parts).shuffle(seed=42)

Filter: 100%|██████████| 6913/6913 [00:00<00:00, 33414.76 examples/s]


In [25]:
#CREAR ENTRENAMIENTO

#eval_split = "validation" if "validation" in tokenized_dataset else "test"

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
#EVALUACION DEL MODELO
results = trainer.evaluate()
results

In [49]:
trainer.save_model("../models/robertuito_finetuned")
tokenizer.save_pretrained("../models/robertuito_finetuned")

Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.17s/it]


('../models/robertuito_finetuned\\tokenizer_config.json',
 '../models/robertuito_finetuned\\tokenizer.json')

EVALUACION DE LOS MODELOS

In [33]:
#SIN FINETUNED
from transformers import pipeline
from sklearn.metrics import classification_report

MODEL_NAME = "pysentimiento/robertuito-sentiment-analysis"

#dataset = load_dataset("mteb/SpanishSentimentClassification")
test_data = dataset["test"]

classifier = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME
)

texts = test_data["text"][:500]
y_true = test_data["label"][:500]

preds = classifier(texts, truncation=True, max_length=128)

label_map = {
    "NEG": 0,
    "NEU": 1,
    "POS": 2
}

y_pred = [label_map[p["label"]] for p in preds]

print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    target_names=["NEG", "NEU", "POS"],
    zero_division=0
))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2173.77it/s]


              precision    recall  f1-score   support

         NEG       0.50      0.90      0.64        31
         NEU       0.30      0.21      0.25        28
         POS       0.98      0.94      0.96       441

    accuracy                           0.90       500
   macro avg       0.59      0.69      0.62       500
weighted avg       0.91      0.90      0.90       500



In [51]:
#CON FINETUNED

from transformers import pipeline
from sklearn.metrics import classification_report

MODEL_PATH = "../models/robertuito_finetuned"


#dataset = load_dataset("mteb/SpanishSentimentClassification")
test_data = dataset["test"]

classifier = pipeline(
    "sentiment-analysis",
    model=MODEL_PATH,
    tokenizer=MODEL_PATH
)

texts = test_data["text"][:500]
y_true = test_data["label"][:500]

preds = classifier(texts, truncation=True, max_length=128)

label_map = {
    "NEG": 0,
    "NEU": 1,
    "POS": 2
}

y_pred = [label_map[p["label"]] for p in preds]

print(classification_report(y_true, y_pred))


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2257.84it/s]


              precision    recall  f1-score   support

           0       0.76      0.81      0.78        31
           1       0.33      0.71      0.45        28
           2       1.00      0.92      0.95       441

    accuracy                           0.90       500
   macro avg       0.69      0.81      0.73       500
weighted avg       0.94      0.90      0.91       500

